# Assignment 5 - Reranker and Consolidator

## FP3 Not in Context - Consolidation strategy Limitations
Documents with the answer were retrieved from the database but did not make it into the context for generating an answer. This occurs when many documents are returned from the database and a consolidation process takes place to retrieve the answer.


## Goal
This notebook will address the reranking and consolidation steps of the RAG system

### Reranking
Cross Encoder, MMR (Maximal Marginal Relevance), Reciprocal Rank Fusion


### Consolidation
Possible include an LLM call?


In [1]:
%%capture
!pip install -q -U langchain langchain-community langchain-huggingface langchain-qdrant
!pip install -q -U qdrant-client sentence-transformers arxiv pymupdf
!pip install -q -U transformers accelerate bitsandbytes

!pip install -q -U xmltodict

!pip install -q -U cohere

!pip install -q -U langchain-cohere

!pip install -q -U wikipedia

!pip install rouge-score

In [19]:
import os
import numpy as np
import time
import locale
from google.colab import userdata

from google.colab import drive
drive.mount('/content/drive')


# IMPORTANT: Add your Hugging Face token to Colab's Secrets (the key icon on the left panel)
# and name it 'HF_TOKEN', or replace the line below with os.environ["HF_TOKEN"] = "your_token"
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

COHERE_API_KEY = userdata.get('COHERE_API_KEY')

import langchain
from langchain_community.document_loaders import ArxivLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_qdrant import QdrantVectorStore

from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.utils.function_calling import convert_to_openai_tool


from langchain_community.document_loaders import ArxivLoader
from langchain_community.document_loaders import PyMuPDFLoader

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig


from langchain_cohere import ChatCohere

import torch.nn.functional as F
from torch import Tensor
from transformers import AutoModel

import bs4
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import WikipediaLoader

from rouge_score import rouge_scorer


import os
os.environ["USER_AGENT"] = "RAG_Assignment/v0.1 (davidschaaf@berkeley.edu)"

# 2. Restore your data at the start of a new session
!tar -xzf "/content/drive/MyDrive/Colab Data/MIDS-267-A5/qdrant_250_40.tar.gz" -C /content/qdrant_storage/


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/bin/bash: -c: line 1: unexpected EOF while looking for matching `"'
/bin/bash: -c: line 2: syntax error: unexpected end of file


In [3]:
%%capture
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics.pairwise import linear_kernel # dot product
EMBEDDINGS_MODEL = 'sentence-transformers/multi-qa-mpnet-base-dot-v1'
base_embeddings = HuggingFaceEmbeddings(model_name=EMBEDDINGS_MODEL)

In [29]:
vector_store = QdrantVectorStore(
    client=QdrantClient(path="/content/qdrant_storage"),
    embedding=base_embeddings,
    collection_name="rag_tech_db",
    distance=Distance.DOT
)


In [30]:
vector_store.similarity_search("What is the meaning of RAG?")

[Document(metadata={'source': 'https://lilianweng.github.io/posts/2024-07-07-hallucination/', 'split_id': 108, 'doc_num': 42, 'id': 'https://lilianweng.github.io/posts/2024-07-07-hallucination/', '_id': 'c1a468f612a4401bb1c87f1200ac54fc', '_collection_name': 'rag_tech_db'}, page_content='RAG → Edits and Attribution#\nRAG (Retrieval-augmented Generation) is a very common approach to provide grounding information, that is to retrieve relevant documents and then generate with related documents as extra context.'),
 Document(metadata={'title': 'Retrieval-augmented generation', 'summary': 'Retrieval-augmented generation (RAG) is a technique that enables large language models (LLMs) to retrieve and incorporate new information from external data sources. With RAG, LLMs first refer to a specified set of documents, then respond to user queries. These documents supplement information from the LLM\'s pre-existing training data. This allows LLMs to use domain-specific and/or updated information th

In [ ]:
%%capture
quantization_config = BitsAndBytesConfig(
   load_in_4bit=True,
   bnb_4bit_quant_type="nf4",
   bnb_4bit_use_double_quant=True,
   bnb_4bit_compute_dtype=torch.bfloat16
)

llm_mistral_model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.3",
    dtype=torch.float32,
    device_map='auto',
    quantization_config=quantization_config
)

llm_mistral_tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.3")

mistral_pipe = pipeline(
    "text-generation",
    model=llm_mistral_model,
    tokenizer=llm_mistral_tokenizer,
    max_new_tokens=1000,
    temperature=0.6,
    top_p=0.95,
    do_sample=True,
    repetition_penalty=1.2
)

mistral_pipe.model.config.pad_token_id = mistral_pipe.model.config.eos_token_id

mistral_llm_lc = HuggingFacePipeline(pipeline=mistral_pipe)

In [ ]:
marketing_persona = {
    "name": "marketing",
    "description": """This user is a marketer who will ask questions about generative AI in order to better understand the products and the field as a whole.
                      They prefer high level answers that explain concept over technical detail.
                      You will help them find accurate, approved messaging about generative AI features, competitive positioning, and technical capabilities to accelerate content production.""",
}

research_persona = {
    "name": "research",
    "description": """This user is an engineer, who requires detailed technical information when they ask questions.
                      You will help them by writing questions about generative AI concepts, internal system architecture, and implementation details."""
}

llm_template = """Write a question that a {name} professional would ask based on the following context.
{description}

Context:
{context}

Respond only with the question nothing else.
Question:"""
llm_prompt_template = PromptTemplate(template=llm_template, input_variables=["object"])

cohere_chat_model = ChatCohere(cohere_api_key=COHERE_API_KEY)

cohere_chain = llm_prompt_template | cohere_chat_model | StrOutputParser()


In [ ]:
# # given a document from the vector store
# # given a persona
# # create a question based on the context-persona
# # save the document number of the question
# # run a similarity search, check if it pulls the correct document
# import random
# random.seed(42)

# def eval_context_persona(context, persona, persona_name):
#     document_number = context.payload['metadata']['doc_num']
#     persona_context = persona.copy()
#     persona_context['context'] = context.payload['page_content']
#     time.sleep(0.1)
#     question = cohere_chain.invoke(persona_context)
#     vector_store_result = vector_store.vector_store.similarity_search_with_score(question, k=5)
#     question_score = vector_store_result[0][1]
#     question_doc_num = vector_store_result[0][0].metadata['doc_num']
#     recall = [1 if x[0].metadata['doc_num'] == document_number else 0 for x in vector_store_result]
#     return {
#         'persona': persona_name,
#         'question_score': question_score,
#         'question_doc_num': question_doc_num,
#         'recall': recall,
#         'question': question
#     }


# points, _ = vector_store.vector_store.client.scroll(
#     collection_name=vector_store.vector_store.collection_name,
#     limit=1000, with_payload=True)

# output = []
# total_questions = 0
# research_scores = []
# marketing_scores = []
# research_recall_1 = []
# research_recall_3 = []
# research_recall_5 = []
# marketing_recall_1 = []
# marketing_recall_3 = []
# marketing_recall_5 = []

# for _ in range(1):
#     context = random.choice(points)
#     result['context'] = context
#     result['doc_num'] = context.payload['metadata']['doc_num']

#     research_result = eval_context_persona(context, research_persona, 'research')
#     marketing_result = eval_context_persona(context, marketing_persona, 'marketing')
#     for result in [research_result, marketing_result]:
#         result['doc_num'] = context.payload['metadata']['doc_num']
#         result['context_id'] = context.id

#     output.append(research_result)
#     output.append(marketing_result)

#     research_scores.append(research_result['question_score'])
#     marketing_scores.append(marketing_result['question_score'])

#     research_recall_1.append(any(research_result['recall'][:1]))
#     research_recall_3.append(any(research_result['recall'][:3]))
#     research_recall_5.append(any(research_result['recall'][:5]))
#     marketing_recall_1.append(any(marketing_result['recall'][:1]))
#     marketing_recall_3.append(any(marketing_result['recall'][:3]))
#     marketing_recall_5.append(any(marketing_result['recall'][:5]))



# print(f"Average Research Score = {np.mean(research_scores):.1f}")
# print(f"Average Marketing Score = {np.mean(marketing_scores):.1f}")
# print(f"Average Research Recall@1 = {np.mean(research_recall_1):.1f}")
# print(f"Average Research Recall@3 = {np.mean(research_recall_3):.1f}")
# print(f"Average Research Recall@5 = {np.mean(research_recall_5):.1f}")
# print(f"Average Marketing Recall@1 = {np.mean(marketing_recall_1):.1f}")
# print(f"Average Marketing Recall@3 = {np.mean(marketing_recall_3):.1f}")
# print(f"Average Marketing Recall@5 = {np.mean(marketing_recall_5):.1f}")


Average Research Score = 28.6
Average Marketing Score = 28.1
Average Research Recall@1 = 0.8
Average Research Recall@3 = 1.0
Average Research Recall@5 = 1.0
Average Marketing Recall@1 = 0.7
Average Marketing Recall@3 = 0.9
Average Marketing Recall@5 = 0.9


In [ ]:


# import json
# path = f"/content/drive/MyDrive/Colab Data/MIDS-267-A5/vector_store_eval_chunk_{CHUNK_SIZE}_olap_{OVERLAP}_output.json"
# json.dump(output, open(path, "w"), indent=2)

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np

PERSONAS = ["research", "marketing"]


def summarize(records, persona):
    rows = [r for r in records if r.get("persona") == persona]
    if not rows:
        return None

    scores = [r["question_score"] for r in rows]

    # recall@k: did the gold doc appear anywhere in the top k?
    def recall_at(k):
        return np.mean([any(r["recall"][:k]) for r in rows])

    # density: what fraction of the top-5 came from the gold doc?
    density = np.mean([sum(r["recall"]) / len(r["recall"]) for r in rows])

    return {
        "n": len(rows),
        "score": np.mean(scores),
        "r1": recall_at(1),
        "r3": recall_at(3),
        "r5": recall_at(5),
        "density": density,
    }


header = (
    f"{'config':<28}{'persona':<12}{'n':>5}{'score':>8}"
    f"{'R@1':>7}{'R@3':>7}{'R@5':>7}{'top5_density':>14}"
)
print(header)
print("-" * len(header))

paths = [
    "/content/drive/MyDrive/Colab Data/MIDS-267-A5/vector_store_eval_chunk_250_olap_40_output.json",
    "/content/drive/MyDrive/Colab Data/MIDS-267-A5/vector_store_eval_chunk_150_olap_30_output.json"
]

for path in paths:
    records = json.loads(Path(path).read_text())
    config = Path(path).stem

    for persona in PERSONAS:
        m = summarize(records, persona)
        if m is None:
            continue
        print(
            f"{config:<28.28}{persona:<12}{m['n']:>5}{m['score']:>8.1f}"
            f"{m['r1']:>7.2f}{m['r3']:>7.2f}{m['r5']:>7.2f}{m['density']:>14.2f}"
        )


config                      persona         n   score    R@1    R@3    R@5  top5_density
----------------------------------------------------------------------------------------
vector_store_eval_chunk_250_research      100    30.2   0.93   0.97   0.98          0.86
vector_store_eval_chunk_250_marketing     100    27.9   0.68   0.85   0.91          0.67
vector_store_eval_chunk_150_research      100    28.6   0.85   0.96   0.99          0.77
vector_store_eval_chunk_150_marketing     100    28.1   0.68   0.90   0.95          0.64


## Test of Vector Store Hyper Parameters

Tested against both personas - research and marketing.

**Option A:** 250 tokens per chunk and overlap 40, this is the max recommended with the sentence transformers model
**Option B:** 150 tokens per chunk and overlap 30, a reduced size to compare

\>250 was not tested based on the documentation. While it can handle up to 512 tokens, that was not advised and the model was train on chunks up to 250 tokens.

### Procedure
Populate the vector store with hyperparameters chosen for chunk size and overlap.

Select 100 random chunks from the vector store, these are the Original documents.

For each Original docuemnt, use Cohere to generate a question given the context and the persona. Research and marketing each generate 1 question per Original document.

Query the vector store with that Question.

Return the top 5 results analyze the Returned documents.


### Metrics

Score - semantic similarity score of a question and a chunk recalled
Recall@1, Recall@3, Recall@5 - Does the document that was used to generate the question appear in the top N documents of the query based on that question?
Top 5 Density - Out of the top 5 documents, how many come from the original?

### Findings

Research had a clear winner - 250 chunk size, 40 overlap. In both recall and density, the 250 chunks were meaningfully better (Score +1.8, Recall@1 +0.08, Top5Density +0.09).

Marketing was mixed. The score differential was +0.2 for 150/30, and it also won Recall@3 was +0.05, Recall@5 were +0.04 for 150/30. However 250/40 wins on Top5Density (+0.03) and they tie on Recall@1 (both 0.68).

### Conclusion
Given that research had a clear winner (250 chunk size / 40 overlap), and marketing was closer to neutral, I will proceed with 250/40 for the vector store hyperparameters in the RAG system.



